# A/C 개인정보 탐지 지표 검증 Notebook

**목표:** 기존 JSONL 정답/예측을 읽어 지표를 **셀별로 재계산**하고 화면에만 표시합니다. 평가용 CSV·JSON·로그 파일을 **생성하지 않습니다**. 모델 추론·규칙 변경도 하지 않습니다.

- **A:** KoELECTRA V2 단독 / **C:** KoELECTRA V2 + 규칙 기반 선택적 결합
- 평가 단위: `Strict/Partial` = 개인정보 **개체**, `Mask Recall` = 개인정보 **문자**
- 원본 텍스트·개인정보 내용은 화면에 출력하지 않습니다.
- **중요:** Dev는 저장된 `A/C/predictions.jsonl`을 읽어 재계산할 수 있습니다. Test 중간 평가에서는 **집계 JSON만** 생성되었으므로, Test A/C 예측 JSONL이 별도로 없다면 이 노트북으로 Test를 재계산할 수 없습니다. 그 경우 프로젝트의 기존 V2/C 함수를 **메모리에서 호출하여 예측을 구성**하도록 Codex에서 연계해야 합니다. 집계 JSON을 읽는 것은 재계산이 아닙니다.

## 1. 경로 및 평가 대상 설정
`SPLIT`을 `dev` 또는 `test`로 설정합니다. 프로젝트 루트는 `run_compare.py`로 자동 탐색하며, 필요하면 직접 지정합니다.

In [ ]:
from pathlib import Path
import json
from collections import Counter, defaultdict
import pandas as pd
from IPython.display import display

SPLIT = "dev"  # "dev" 또는 "test"
PROJECT_ROOT = None  # 자동 탐색 실패 시: Path(r"C:\Users\USER\NLP_06\rag_chatbot\nlp_like_lion_chatbot_03")

# 기본 예측 경로: Dev에서 저장된 A/C 예측에 맞춤
PRED_A_OVERRIDE = None  # 예: Path("...")
PRED_C_OVERRIDE = None
GOLD_OVERRIDE = None

if PROJECT_ROOT is None:
    search_roots = [Path.cwd(), *Path.cwd().parents]
    found = next((p for p in search_roots if (p / "run_compare.py").is_file()), None)
    if found is None:
        raise FileNotFoundError("프로젝트 루트를 찾지 못했습니다. PROJECT_ROOT를 지정하세요.")
    PROJECT_ROOT = found
else:
    PROJECT_ROOT = Path(PROJECT_ROOT).expanduser().resolve()

GOLD_FILE = Path(GOLD_OVERRIDE) if GOLD_OVERRIDE else PROJECT_ROOT / "data" / SPLIT / f"pii_synth_finance_{SPLIT}.jsonl"
PRED_A_FILE = Path(PRED_A_OVERRIDE) if PRED_A_OVERRIDE else PROJECT_ROOT / "local_results" / "ensemble_experiment" / SPLIT / "A" / "predictions.jsonl"
PRED_C_FILE = Path(PRED_C_OVERRIDE) if PRED_C_OVERRIDE else PROJECT_ROOT / "local_results" / "ensemble_experiment" / SPLIT / "C" / "predictions.jsonl"

for name, f in {"Gold": GOLD_FILE, "A 예측": PRED_A_FILE, "C 예측": PRED_C_FILE}.items():
    print(f"{name:6}: {'존재' if f.is_file() else '없음'} | {f}")

## 2. Gold / A / C 예측 읽기
개체에는 `label`, `start`, `end`가 있으며 문자 구간은 `[start, end)` 기준입니다. 저장된 예측이 없는 경우 **임의 수치로 대체하지 않고 종료**합니다.

In [ ]:
LABELS = ["person", "phone_number", "email", "address",
          "resident_registration_number", "account_number", "card_number"]

def load_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

missing_files = [str(p) for p in (GOLD_FILE, PRED_A_FILE, PRED_C_FILE) if not p.is_file()]
if missing_files:
    raise FileNotFoundError(
        "재계산에 필요한 JSONL이 없습니다:\n" + "\n".join(missing_files) +
        "\nTest의 집계 evaluation.json만으로는 개체·문자 지표를 재계산할 수 없습니다. "
        "Test A/C 예측을 기존 파이프라인으로 메모리에서 구성한 뒤 이 셀에 연결해야 합니다."
    )

docs = load_jsonl(GOLD_FILE)
assert len({d["id"] for d in docs}) == len(docs), "Gold 문서 ID 중복"
for d in docs:
    d["gold"] = [e for e in d["entities"] if e["label"] in LABELS]
    for g in d["gold"]:
        if "text" in g:
            assert d["text"][g["start"]:g["end"]] == g["text"], "Gold span 무결성 오류"

def load_predictions(path, docs):
    rows = load_jsonl(path)
    by_id = {r["id"]: r for r in rows}
    assert len(by_id) == len(rows), "예측 문서 ID 중복"
    doc_ids = {d["id"] for d in docs}
    missing = doc_ids - by_id.keys()
    extra = by_id.keys() - doc_ids
    if missing or extra:
        raise ValueError(f"Gold/예측 문서 불일치: 누락 {len(missing)}개, 추가 {len(extra)}개")
    out = []
    for d in docs:
        ents = []
        for e in by_id[d["id"]].get("entities", []):
            s, t = int(e["start"]), int(e["end"])
            if not (0 <= s < t <= len(d["text"])):
                raise ValueError(f"잘못된 span 범위: {d['id']}")
            ents.append({"label": e.get("label"), "start": s, "end": t})
        out.append(ents)
    return out

preds = {"A": load_predictions(PRED_A_FILE, docs), "C": load_predictions(PRED_C_FILE, docs)}
print(f"{SPLIT.upper()} | 문서 {len(docs):,}개 | Gold {sum(len(d['gold']) for d in docs):,}개")
for name, arr in preds.items():
    print(f"{name} | 예측 개체 {sum(len(p) for p in arr):,}개")

## 3. Strict / Partial / Mask Recall 계산 함수
`run_compare.py`의 **개체 1:1 Partial 매칭**·Strict span 완전 일치·문자 단위 Mask Recall 정의를 그대로 따릅니다. 비율 분모가 0이면 공식 평가 코드와 동일하게 0으로 둡니다.

In [ ]:
def overlap(a, b):
    return a["start"] < b["end"] and b["start"] < a["end"]

def key(e):
    return (e["label"], e["start"], e["end"])

def prf(tp, n_pred, n_gold):
    p = tp / n_pred if n_pred else 0.0
    r = tp / n_gold if n_gold else 0.0
    return p, r, (2 * p * r / (p + r) if p + r else 0.0)

def official_counts(docs, predicted):
    """첨부 run_compare.py의 count()와 동일한 집계 규칙."""
    c = defaultdict(Counter)
    for d, P in zip(docs, predicted):
        G = d["gold"]
        pkeys = {key(p) for p in P}
        for lab in LABELS:
            g = [x for x in G if x["label"] == lab]
            p = [x for x in P if x["label"] == lab]
            k = c[lab]
            k["gold"] += len(g)
            k["pred"] += len(p)
            k["strict_tp"] += sum(key(x) in pkeys for x in g)
            used = set()
            for x in g:
                for j, y in enumerate(p):
                    if j not in used and overlap(x, y):
                        used.add(j)
                        k["partial_tp"] += 1
                        break
            for x in g:
                cov = set()
                for y in P:
                    cov.update(range(max(x["start"], y["start"]), min(x["end"], y["end"])))
                k["chars"] += x["end"] - x["start"]
                k["chars_masked"] += len(cov)
    return c

def derive(c):
    t = Counter()
    for v in c.values():
        t.update(v)
    sp, sr, sf = prf(t["strict_tp"], t["pred"], t["gold"])
    pp, pr, pf = prf(t["partial_tp"], t["pred"], t["gold"])
    return dict(gold=t["gold"], pred=t["pred"],
                strict_tp=t["strict_tp"], strict_fp=t["pred"] - t["strict_tp"],
                strict_fn=t["gold"] - t["strict_tp"],
                partial_tp=t["partial_tp"], partial_fp=t["pred"] - t["partial_tp"],
                partial_fn=t["gold"] - t["partial_tp"],
                strict_p=100*sp, strict_r=100*sr, strict_f1=100*sf,
                partial_p=100*pp, partial_r=100*pr, partial_f1=100*pf,
                mask_rec=100*t["chars_masked"] / t["chars"] if t["chars"] else 0,
                gold_chars=t["chars"], masked_gold_chars=t["chars_masked"],
                unmasked_chars=t["chars"]-t["chars_masked"])

counts = {name: official_counts(docs, arr) for name, arr in preds.items()}
summary = {name: derive(c) for name, c in counts.items()}
print("평가 함수 준비 완료 (파일 저장 없음)")

## 4. 전체 Strict / Partial 지표
**FP는 문서 수가 아니라 오탐 예측 개체 수**입니다. Dev와 Test 규모가 다르므로 비율 지표(Precision/Recall/F1)를 함께 보세요.

In [ ]:
cols = ["gold", "pred", "strict_tp", "strict_fp", "strict_fn", "strict_p", "strict_r", "strict_f1",
        "partial_tp", "partial_fp", "partial_fn", "partial_p", "partial_r", "partial_f1"]
table = pd.DataFrame.from_dict(summary, orient="index")[cols]
table.index.name = "모델"
display(table.round(3))

## 5. 개인정보 보호 지표
Gold 개체의 모든 문자가 **라벨과 관계없이** 예측 구간에 덮이면 *완전 보호*, 하나도 겹치지 않으면 *완전 미탐*입니다. **비중첩 오탐**은 어떠한 Gold 개체와도 겹치지 않는 예측 개체이고, **정답 외 마스킹 문자**는 모든 Gold span 바깥에서 마스킹되는 고유 문자 위치 수입니다.

In [ ]:
def char_positions(entities):
    return {i for e in entities for i in range(e["start"], e["end"])}

def doc_protection(d, P):
    G = d["gold"]
    gold_positions = char_positions(G)
    pred_positions = char_positions(P)
    return {
        "gold_char_set": gold_positions,
        "pred_char_set": pred_positions,
        "fully_protected": sum(set(range(g["start"],g["end"])) <= pred_positions for g in G),
        "fully_missed": sum(not any(overlap(g,p) for p in P) for g in G),
        "nonoverlap_fp": sum(not any(overlap(p,g) for g in G) for p in P),
        "unmasked_chars": len(gold_positions - pred_positions),
        "nongold_masked": len(pred_positions - gold_positions),
        "document_has_leak": int(bool(gold_positions - pred_positions)),
    }

protection = {}
protection_docs = {}
for name, arr in preds.items():
    stats = [doc_protection(d, P) for d, P in zip(docs,arr)]
    protection_docs[name] = stats
    protection[name] = {
        "개체 완전 보호": sum(x["fully_protected"] for x in stats),
        "개체 완전 보호율(%)": 100 * sum(x["fully_protected"] for x in stats) / sum(len(d["gold"]) for d in docs),
        "완전 미탐": sum(x["fully_missed"] for x in stats),
        "미보호 문자": sum(x["unmasked_chars"] for x in stats),
        "Mask Recall(%)": summary[name]["mask_rec"],
        "비중첩 FP": sum(x["nonoverlap_fp"] for x in stats),
        "정답 외 마스킹 문자": sum(x["nongold_masked"] for x in stats),
        "미보호 문자 존재 문서": sum(x["document_has_leak"] for x in stats),
        "미보호 문자 존재 문서율(%)": 100*sum(x["document_has_leak"] for x in stats)/len(docs),
    }
display(pd.DataFrame.from_dict(protection,orient="index").round(3))

## 6. 7종 라벨별 지표
같은 라벨에서 Strict / Partial Precision, Recall, F1, 개체 완전 보호율, 완전 미탐, 미보호 문자 수를 비교합니다. Mask Recall 및 완전 보호는 **다른 라벨의 예측이 덮더라도 보호된 것**으로 계산합니다.

In [ ]:
rows = []
for name, P_all in preds.items():
    for lab in LABELS:
        c = counts[name][lab]
        sp,sr,sf = prf(c["strict_tp"], c["pred"], c["gold"])
        pp,pr,pf = prf(c["partial_tp"], c["pred"], c["gold"])
        full = missed = uncovered = 0
        for d, P in zip(docs,P_all):
            covered = char_positions(P)
            for g in d["gold"]:
                if g["label"] != lab:
                    continue
                indices = set(range(g["start"],g["end"]))
                full += int(indices <= covered)
                missed += int(not any(overlap(g,p) for p in P))
                uncovered += len(indices-covered)
        rows.append(dict(모델=name, 라벨=lab, Gold=c["gold"], 예측=c["pred"], Strict_FP=c["pred"]-c["strict_tp"],
                         Strict_P=100*sp, Strict_R=100*sr, Strict_F1=100*sf,
                         Partial_P=100*pp, Partial_R=100*pr, Partial_F1=100*pf,
                         Mask_Recall=100*c["chars_masked"]/c["chars"] if c["chars"] else 0,
                         완전보호=full, 완전보호율=100*full/c["gold"] if c["gold"] else 0,
                         완전미탐=missed, 미보호문자=uncovered))
label_df=pd.DataFrame(rows).set_index(["라벨","모델"]).sort_index()
display(label_df.round(3))

## 7. A → C의 변화량
개선 효과와 부작용을 같은 Gold 기준으로 비교합니다. 숫자 차이는 `C − A`이며 비율 차이는 **%p**입니다. **추가 보호된 정답 문자**와 **정답 외 추가 마스킹 문자**는 기존 A 대비 문자 위치의 **집합 차이**로 계산합니다.

In [ ]:
compare_cols = ["strict_p", "strict_r", "strict_f1", "partial_p", "partial_r", "partial_f1",
                "mask_rec", "strict_fp", "partial_fp", "unmasked_chars"]
rows = []
for k in compare_cols:
    rows.append({"지표":k, "A":summary["A"][k], "C":summary["C"][k], "C-A":summary["C"][k]-summary["A"][k]})
for k in ["개체 완전 보호", "개체 완전 보호율(%)", "완전 미탐", "비중첩 FP", "정답 외 마스킹 문자", "미보호 문자 존재 문서"]:
    rows.append({"지표":k, "A":protection["A"][k], "C":protection["C"][k], "C-A":protection["C"][k]-protection["A"][k]})
display(pd.DataFrame(rows).set_index("지표").round(3))

added_gold = lost_gold = added_nongold = lost_nongold = 0
for sa, sc in zip(protection_docs["A"], protection_docs["C"]):
    gold = sa["gold_char_set"]
    a = sa["pred_char_set"]
    c = sc["pred_char_set"]
    added_gold += len((c-a) & gold)
    lost_gold += len((a-c) & gold)
    added_nongold += len((c-a)-gold)
    lost_nongold += len((a-c)-gold)
print(f"C가 A 대비 새로 보호한 Gold 문자: {added_gold}자")
print(f"C가 A 대비 잃은 Gold 보호 문자: {lost_gold}자")
print(f"C의 정답 외 신규 마스킹: {added_nongold}자")
print(f"C에서 제거된 정답 외 마스킹: {lost_nongold}자")

## 8. 문서별 문제 발생 여부 (PII 텍스트 미출력)
문서 ID와 **집계 수치만** 출력합니다. 원문·부분 원문·PII 문자열은 출력하지 않습니다.

In [ ]:
detail=[]
for i,d in enumerate(docs):
    a = protection_docs["A"][i]
    c = protection_docs["C"][i]
    detail.append(dict(doc_id=d["id"], gold_count=len(d["gold"]),
                       A_unmasked_chars=a["unmasked_chars"], C_unmasked_chars=c["unmasked_chars"],
                       A_fully_missed=a["fully_missed"], C_fully_missed=c["fully_missed"],
                       newly_protected_chars=len((c["pred_char_set"]-a["pred_char_set"]) & a["gold_char_set"])))
doc_df=pd.DataFrame(detail)
display(doc_df.loc[(doc_df["A_unmasked_chars"] > 0) | (doc_df["C_unmasked_chars"] > 0) | (doc_df["newly_protected_chars"] > 0)]
        .sort_values(["C_unmasked_chars","A_unmasked_chars"],ascending=False)
        .reset_index(drop=True))

## 9. 참고값과 대조 (검증용)
대화에서 공유하신 결과와 숫자가 일치하는지 확인합니다. **검증용 기준값은 지표 계산에 사용되지 않습니다.** 다른 데이터나 버전을 쓰는 경우 불일치가 정상일 수 있습니다.

In [ ]:
EXPECTED = {
  "dev": {
    "A": {"strict_f1":92.007, "mask_rec":99.850, "strict_fp":60, "unmasked_chars":7},
    "C": {"strict_f1":92.103, "mask_rec":99.914, "strict_fp":60, "unmasked_chars":4},
  },
  "test": {
    "A": {"strict_f1":89.448, "mask_rec":99.528, "strict_fp":176, "unmasked_chars":44},
    "C": {"strict_f1":89.703, "mask_rec":99.807, "strict_fp":174, "unmasked_chars":18},
  }
}
for name in ("A","C"):
    for metric, expected in EXPECTED.get(SPLIT,{}).get(name,{}).items():
        actual = summary[name][metric]
        tol = 0.011 if isinstance(expected,float) else 0
        ok = abs(actual-expected) <= tol
        print(f"{'일치' if ok else '차이'} | {name} {metric}: 계산={actual:.3f}, 참고={expected}")

print("완료: 새로운 평가 파일을 저장하지 않았습니다. 단, Notebook 자체의 셀 출력은 저장 시 .ipynb에 포함될 수 있습니다.")